# Manualisasi
Subbab 4.11. Menghitung ulang seluruh angka manualisasi dengan model Whisper contoh, mulai dari spektrogram log-Mel sampai urutan kata, lalu mengevaluasinya dengan keempat metrik.

```
[1] Spektrogram masukan X         -> data/output/manualisasi/X_spektrogram.json
[2] Model contoh                  -> bobot dan fungsi perhitungan (di memori)
[3] Verifikasi terhadap naskah    -> X dan bobot versi lama harus mereproduksi angka lama
[4] Langkah 1–7: alur komputasi   -> data/output/manualisasi/langkah_1_7.json, beam_search.csv
[5] Langkah 8–11: metrik          -> data/output/manualisasi/penyelarasan.csv, metrik.json
[6] Langkah 12: RTF               -> data/output/manualisasi/rtf.csv
[7] Metadata run                  -> data/output/manualisasi/run_meta_manualisasi.json
```

Spektrogram X diambil dari segmen rekaman uji AOK-04 yang sama dengan transkrip acuan pada langkah 8. Bobot model contoh dipilih sendiri, bukan bobot Whisper hasil pelatihan, sehingga setiap langkah dapat dihitung dengan kalkulator.

Bagian 1 membutuhkan `data/processed/audio/AOK-04.wav`. Bila file itu tidak ada, X dibaca dari `X_spektrogram.json` hasil run sebelumnya.

## 0. Persiapan

In [1]:
# !pip install faster-whisper==1.2.1 jiwer==4.0.0 soundfile numpy pandas

In [2]:
from pathlib import Path
import sys, json, math, datetime, importlib, platform
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.praproses_teks as pt
import src.metrik as mt
importlib.reload(pt); importlib.reload(mt)

OUT        = ROOT / "data" / "output"
MAN        = OUT / "manualisasi"
AUDIO_PROC = ROOT / "data" / "processed" / "audio"
GLOSARIUM  = ROOT / "data" / "processed" / "glosarium" / "glosarium_semua.csv"
KAMUS_PATH = ROOT / "data" / "processed" / "kamus_normalisasi.json"
MASTER     = OUT / "master"
LOG_WAKTU  = OUT / "tahap1" / "waktu_transkripsi.jsonl"
MAN.mkdir(parents=True, exist_ok=True)

rel = lambda p: Path(p).relative_to(ROOT).as_posix()

c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
FID, MULAI, SELESAI = "AOK-04", 633.64, 640.50   # segmen acuan yang dievaluasi pada langkah 8
MATA_KULIAH = "AOK"
PITA = [10, 40]          # pita ke-11 dan ke-41 dari 80 pita Mel
DESIMAL_X = 1            # X dibulatkan satu desimal agar mudah dihitung manual
VARIAN_RTF = "large-v3"  # varian terpilih Tahap 1, dipakai untuk contoh RTF

def koma(M, d=4):
    """Cetak angka dengan koma desimal, siap disalin ke naskah."""
    M = np.atleast_2d(np.asarray(M, dtype=float))
    return "\n".join("  ".join(f"{v:.{d}f}".replace(".", ",") for v in baris) for baris in M)

def tampil(nama, M, d=4):
    print(f"{nama}:\n{koma(M, d)}\n")

## 1. Spektrogram masukan X
Spektrogram log-Mel segmen dihitung dengan ekstraktor fitur faster-whisper, yaitu masukan yang sama dengan yang diterima Whisper. Segmen 6,86 detik menghasilkan 80 pita × 687 *frame*, terlalu besar untuk dihitung manual. Yang diambil hanya 2 pita × 4 *frame* pada bagian yang paling keras, yaitu bagian yang berisi suara, lalu dibulatkan satu desimal.

In [4]:
X_JSON = MAN / "X_spektrogram.json"
wav = AUDIO_PROC / f"{FID}.wav"

if wav.exists():
    import soundfile as sf
    from faster_whisper.feature_extractor import FeatureExtractor
    audio, sr = sf.read(wav, dtype="float32")
    assert sr == 16000, f"sampling rate {sr}"
    seg = audio[int(MULAI * sr):int(SELESAI * sr)]
    mel = FeatureExtractor(feature_size=80)(seg)
    energi = mel.mean(axis=0)
    f0 = int(np.argmax(energi[:-4]))
    X_asli = mel[PITA, f0:f0 + 4]
    info_x = {"rekaman": FID, "segmen_detik": [MULAI, SELESAI], "ukuran_mel": list(mel.shape),
              "pita": PITA, "frame_mulai": f0, "detik_mulai": round(MULAI + f0 * 0.01, 2),
              "X": np.round(X_asli, 4).tolist(), "X_dibulatkan": np.round(X_asli, DESIMAL_X).tolist()}
    X_JSON.write_text(json.dumps(info_x, indent=2), encoding="utf-8")
else:
    assert X_JSON.exists(), f"{rel(wav)} dan {rel(X_JSON)} sama-sama tidak ada"
    info_x = json.loads(X_JSON.read_text(encoding="utf-8"))
    print(f"[!] {rel(wav)} tidak ada, X dibaca dari {rel(X_JSON)}\n")

X = np.array(info_x["X_dibulatkan"], dtype=float) + 0.0   # + 0.0 menghapus -0.0
print("ukuran mel:", info_x["ukuran_mel"])
print("frame mulai:", info_x["frame_mulai"], "→ detik", info_x["detik_mulai"])
tampil("X (4 desimal)", info_x["X"])
tampil("X yang dipakai", X, 1)

ukuran mel: [80, 687]
frame mulai: 393 → detik 637.57
X (4 desimal):
0,5503  0,5776  0,5107  0,4167
-0,0983  0,0006  -0,1946  -0,2693

X yang dipakai:
0,6  0,6  0,5  0,4
-0,1  0,0  -0,2  -0,3



## 2. Model contoh
Ukuran model: 2 pita Mel, 4 *frame*, d_model = 4, satu *head*, satu lapis *encoder*, satu lapis *decoder*, lebar lapis tersembunyi MLP 8, dan kosakata 4 token (SOT, *main*, *memory*, EOT).

Ketentuan perhitungan sama dengan naskah:
- semua bias 0, parameter g dan b pada LN bernilai 1 dan 0, ε diabaikan;
- setiap nilai antara dibulatkan empat desimal (setengah ke atas, seperti kalkulator), dan langkah berikutnya memakai nilai yang sudah dibulatkan;
- Φ pada GELU dibulatkan empat desimal sebelum dikalikan.

Dengan X dari rekaman, bobot *embedding* versi lama menghasilkan keluaran SOT SOT. Tiga nilai pada W_emb karena itu disetel ulang, dan nilainya tetap sederhana. Bobot lain tidak berubah.

In [5]:
def bulat(x):
    if not np.isfinite(x):
        return x
    return float(Decimal(repr(float(x))).quantize(Decimal("0.0001"), rounding=ROUND_HALF_UP))

_bulat = np.vectorize(bulat)

def R(a):
    a = np.asarray(a, dtype=float)
    return _bulat(a) if a.size else a

def Phi(x):
    return bulat(0.5 * (1 + math.erf(x / math.sqrt(2))))

def gelu(a):
    a = np.asarray(a, dtype=float)
    return R(a * np.vectorize(Phi)(a))

def ln_rinci(baris):
    mu = bulat(baris.mean())
    dev = R(baris - mu)
    jumlah = bulat(float((dev ** 2).sum()))
    var = bulat(jumlah / len(baris))
    sd = bulat(math.sqrt(var))
    return {"mu": mu, "dev": dev, "jumlah_kuadrat": jumlah, "var": var, "sd": sd, "hasil": R(dev / sd)}

def ln(M):
    return np.array([ln_rinci(b)["hasil"] for b in np.atleast_2d(M)])

def softmax(S):
    out = []
    for baris in np.atleast_2d(S):
        e = R(np.exp(baris))
        out.append(R(e / e.sum()))
    return np.array(out)

def mlp(x):
    x = np.atleast_2d(x)
    return R(gelu(x) + R(0.5 * gelu(-x)))       # W1 = [I  -I], W2 = [I ; 0,5 I]

def attention(Q, K, V, mask=False):
    S = R(R(Q @ K.T) / 2)                       # dibagi akar d_k = 2
    if mask:
        S = S.copy()
        S[np.triu_indices_from(S, k=1)] = -np.inf
    A = softmax(S)
    return S, A, R(A @ V)

In [6]:
W_CONV1 = [np.array([[0.5, 1, 0], [0, 0, 0.5]]), np.array([[0, 0, 0], [0.5, 1, 0.5]]),
           np.array([[1, 0, -1], [0, 0, 0]]),    np.array([[0, 0.5, 0], [0, -0.5, 0]])]
K_CONV2 = {0: [0, 1, 0], 1: [0, 1, 0], 2: [0.5, 1, 0.5], 3: [0, 1, 0]}   # kanal keluaran: bobot pada kanal yang sama
W_Q = np.diag([0.5, -1, 1, 1])
W_K = np.diag([0.5, 1, 1, -0.5])
W_V = np.diag([-1.0, -1.0, -1.0, -1.0])                                  # W_O = identitas
PE_D = np.array([[-0.5, 0, 0.5, -0.5], [-0.5, -0.5, 0, 0.5]])
TOKEN = ["SOT", "main", "memory", "EOT"]

W_EMB_LAMA = np.array([[ 0.5, -0.5,  0.5, 1.0],
                       [ 1.0,  0.0, -0.5, 0.0],
                       [-1.0, -0.5,  1.0, 0.0],
                       [-1.0, -1.0,  1.0, 0.0]])
W_EMB = W_EMB_LAMA.copy()
W_EMB[0, 1] = 0.5    # SOT
W_EMB[2, 0] = 0.5    # memory
W_EMB[3, 1] = 0.0    # EOT

pd.DataFrame({"lama": [koma(r, 1) for r in W_EMB_LAMA], "baru": [koma(r, 1) for r in W_EMB]}, index=TOKEN)

,lama,baru
SOT,"0,5 -0,5 0,5 1,0","0,5 0,5 0,5 1,0"
main,"1,0 0,0 -0,5 0,0","1,0 0,0 -0,5 0,0"
memory,"-1,0 -0,5 1,0 0,0","0,5 -0,5 1,0 0,0"
EOT,"-1,0 -1,0 1,0 0,0","-1,0 0,0 1,0 0,0"


In [7]:
def encoder(X):
    Xp = np.pad(X, ((0, 0), (1, 1)))                                     # padding satu frame nol di tiap tepi
    conv1 = R([[(W_CONV1[k] * Xp[:, t:t + 3]).sum() for t in range(4)] for k in range(4)])
    gelu1 = gelu(conv1)
    gp = np.pad(gelu1, ((0, 0), (1, 1)))
    conv2 = R([[np.dot(K_CONV2[c], gp[c, 2 * t:2 * t + 3]) for t in range(2)] for c in range(4)])   # stride 2
    gelu2 = gelu(conv2)
    PE = R([[math.sin(p), math.sin(p * 1e-4), math.cos(p), math.cos(p * 1e-4)] for p in range(2)])
    Z0 = R(gelu2.T + PE)
    LZ = ln(Z0)
    Q, K, V = R(LZ @ W_Q), R(LZ @ W_K), R(LZ @ W_V)
    S, A, MHA = attention(Q, K, V)
    x1 = R(Z0 + MHA)
    Lx1 = ln(x1)
    M = mlp(Lx1)
    x2 = R(x1 + M)
    H = ln(x2)
    return dict(conv1=conv1, gelu1=gelu1, conv2=conv2, gelu2=gelu2, PE=PE, Z0=Z0, LN_Z0=LZ, Q=Q, K=K, V=V,
                S=S, A=A, MHA=MHA, x1=x1, LN_x1=Lx1, MLP=M, x2=x2, H=H)

def decoder(token, H, W_emb):
    y = R(W_emb[token] + PE_D[:len(token)])
    Ly = ln(y)
    _, _, sa = attention(R(Ly @ W_Q), R(Ly @ W_K), R(Ly @ W_V), mask=True)
    y1 = R(y + sa)
    q = R(ln(y1) @ W_Q)
    s, a, ca = attention(q, R(H @ W_K), R(H @ W_V))                      # cross-attention ke H
    y2 = R(y1 + ca)
    m = mlp(ln(y2))
    y3 = R(y2 + m)
    Ly3 = ln(y3)
    z = R(Ly3 @ W_emb.T)
    return dict(y=y, LN_y=Ly, v=R(Ly @ W_V), y1=y1, q=q, s=s, a=a, ca=ca, y2=y2, mlp=m, y3=y3,
                LN_y3=Ly3, z=z, p=softmax(z))

def beam(H, W_emb, lebar=2):
    p1 = decoder([0], H, W_emb)["p"][0]
    simpan = list(np.argsort(-p1)[:lebar])
    baris = []
    for t1 in simpan:
        p2 = decoder([0, t1], H, W_emb)["p"][1]
        for t2 in range(4):
            lp1, lp2 = bulat(math.log(p1[t1])), bulat(math.log(p2[t2]))
            baris.append({"deret": f"{TOKEN[t1]} {TOKEN[t2]}", "log_p_1": lp1, "p_2": p2[t2],
                          "log_p_2": lp2, "skor": bulat(lp1 + lp2)})
    return [TOKEN[t] for t in simpan], pd.DataFrame(baris)

## 3. Verifikasi terhadap naskah
Fungsi di atas dijalankan dulu dengan X dan W_emb versi lama. Hasilnya harus sama dengan angka di naskah lama, dengan selisih paling banyak 0,0002. Selisih 1 pada desimal keempat bisa muncul karena urutan pembulatan di tengah jalan.

In [8]:
X_LAMA = np.array([[0.5, 1.0, 0.0, -0.5], [0.0, 0.5, 1.0, 0.5]])
NASKAH_LAMA = {
    "GELU Conv1 kanal 1 frame 2": 1.6798, "Z0 [1,1]": 0.4171, "LN(Z0) [1,1]": -0.5556,
    "a11 encoder": 0.1615, "H [1,4]": 1.6859, "bobot cross-attention frame 1": 0.8462,
    "p(main)": 0.6071, "p(memory)": 0.2156, "skor main memory": -1.1340, "skor main EOT": -1.5951,
}
e0 = encoder(X_LAMA)
d0 = decoder([0], e0["H"], W_EMB_LAMA)
_, b0 = beam(e0["H"], W_EMB_LAMA)
skor0 = b0.set_index("deret").skor
hitungan = [e0["gelu1"][0, 1], e0["Z0"][0, 0], e0["LN_Z0"][0, 0], e0["A"][0, 0], e0["H"][0, 3], d0["a"][0, 0],
            d0["p"][0, 1], d0["p"][0, 2], skor0["main memory"], skor0["main EOT"]]
ver = pd.DataFrame({"naskah_lama": NASKAH_LAMA.values(), "hitungan": hitungan}, index=NASKAH_LAMA.keys())
ver["selisih"] = (ver.hitungan - ver.naskah_lama).abs().round(4)
display(ver)
assert (ver.selisih <= 0.0002).all(), "fungsi tidak mereproduksi angka naskah lama"
print("Verifikasi lolos.")

,naskah_lama,hitungan,selisih
GELU Conv1 kanal 1 frame 2,1.6798,1.6798,0.0
"Z0 [1,1]",0.4171,0.4171,0.0
"LN(Z0) [1,1]",-0.5556,-0.5556,0.0
a11 encoder,0.1615,0.1615,0.0
"H [1,4]",1.6859,1.6859,0.0
bobot cross-attention frame 1,0.8462,0.8462,0.0
p(main),0.6071,0.6071,0.0
p(memory),0.2156,0.2156,0.0
skor main memory,-1.1340,-1.1340,0.0
skor main EOT,-1.5951,-1.5951,0.0


Verifikasi lolos.


## 4. Langkah 1–7: alur komputasi
Seluruh matriks dicetak dengan koma desimal agar dapat langsung disalin ke naskah. Contoh perhitungan satu elemen dicetak pada langkah yang sama dengan naskah.

In [9]:
enc = encoder(X)
dec = decoder([0], enc["H"], W_EMB)
simpan, tabel_beam = beam(enc["H"], W_EMB)

Xp = np.pad(X, ((0, 0), (1, 1)))
print("1. KONVOLUSI PERTAMA")
suku = " + ".join(f"{w:g}({x:g})" for w, x in zip(W_CONV1[0].ravel(), Xp[:, 1:4].ravel()))
print(f"y1(2) = {suku} = {enc['conv1'][0, 1]:g}".replace(".", ","))
print(f"GELU({enc['conv1'][0, 1]:g}) = {enc['conv1'][0, 1]:g} × {Phi(enc['conv1'][0, 1])} = {enc['gelu1'][0, 1]}\n".replace(".", ","))
tampil("Conv1(X)", enc["conv1"]); tampil("GELU(Conv1(X))", enc["gelu1"])

print("2. KONVOLUSI KEDUA")
g = enc["gelu1"][2]
print(f"y3(1) = 0,5(0) + 1({g[0]}) + 0,5({g[1]}) = {enc['conv2'][2, 0]}".replace(".", ","))
print(f"GELU({enc['conv2'][2, 0]}) = {enc['conv2'][2, 0]} × {Phi(enc['conv2'][2, 0])} = {enc['gelu2'][2, 0]}\n".replace(".", ","))
tampil("Conv2", enc["conv2"]); tampil("GELU(Conv2)", enc["gelu2"])

print("3. POSITIONAL ENCODING")
tampil("PE", enc["PE"]); tampil("Z0", enc["Z0"])

1. KONVOLUSI PERTAMA
y1(2) = 0,5(0,6) + 1(0,6) + 0(0,5) + 0(-0,1) + 0(0) + 0,5(-0,2) = 0,8
GELU(0,8) = 0,8 × 0,7881 = 0,6305

Conv1(X):
0,6000  0,8000  0,6500  0,6500
-0,1000  -0,1500  -0,3500  -0,4000
-0,6000  0,1000  0,2000  0,5000
0,3500  0,3000  0,3500  0,3500

GELU(Conv1(X)):
0,4354  0,6305  0,4824  0,4824
-0,0460  -0,0661  -0,1271  -0,1378
-0,1646  0,0540  0,1159  0,3458
0,2229  0,1854  0,2229  0,2229

2. KONVOLUSI KEDUA
y3(1) = 0,5(0) + 1(-0,1646) + 0,5(0,054) = -0,1376
GELU(-0,1376) = -0,1376 × 0,4453 = -0,0613

Conv2:
0,4354  0,4824
-0,0460  -0,1271
-0,1376  0,3158
0,2229  0,2229

GELU(Conv2):
0,2910  0,3305
-0,0222  -0,0571
-0,0613  0,1970
0,1311  0,1311

3. POSITIONAL ENCODING
PE:
0,0000  0,0000  1,0000  1,0000
0,8415  0,0001  0,5403  1,0000

Z0:
0,2910  -0,0222  0,9387  1,1311
1,1720  -0,0570  0,7373  1,1311



In [10]:
print("4. BLOK ENCODER")
r = ln_rinci(enc["Z0"][0])
print(f"mu = {r['mu']}".replace(".", ","))
print(("jumlah (z - mu)^2 = " + " + ".join(f"({d})^2" for d in r["dev"]) + f" = {r['jumlah_kuadrat']}").replace(".", ","))
print(f"sigma^2 = {r['var']}, sigma = {r['sd']}".replace(".", ","))
print(f"LN(z)_1 = ({enc['Z0'][0, 0]} - {r['mu']}) / {r['sd']} = {r['hasil'][0]}\n".replace(".", ","))
for k in ["LN_Z0", "Q", "K", "V"]:
    tampil(k, enc[k])
q1, k2 = enc["Q"][0], enc["K"][1]
print(("q1 k2^T = " + " + ".join(f"({a})({b})" for a, b in zip(q1, k2)) + f" = {bulat(q1 @ k2)}").replace(".", ","))
print(f"s12 = {bulat(q1 @ k2)} / 2 = {enc['S'][0, 1]}".replace(".", ","))
e = R(np.exp(enc["S"][0]))
print(f"a11 = {e[0]} / ({e[0]} + {e[1]}) = {enc['A'][0, 0]}\n".replace(".", ","))
for k in ["S", "A", "MHA", "x1", "LN_x1"]:
    tampil(k, enc[k])
u = enc["LN_x1"][0, 0]
print(f"MLP contoh: GELU({u}) + 0,5 GELU({-u}) = {gelu([u])[0]} + 0,5({gelu([-u])[0]}) = {enc['MLP'][0, 0]}\n".replace(".", ",").replace("0,5,", "0.5,"))
for k in ["MLP", "x2", "H"]:
    tampil(k, enc[k])

4. BLOK ENCODER
mu = 0,5847
jumlah (z - mu)^2 = (-0,2937)^2 + (-0,6069)^2 + (0,354)^2 + (0,5464)^2 = 0,8785
sigma^2 = 0,2196, sigma = 0,4686
LN(z)_1 = (0,291 - 0,5847) / 0,4686 = -0,6268

LN_Z0:
-0,6268  -1,2951  0,7554  1,1660
0,8631  -1,6263  -0,0174  0,7802

Q:
-0,3134  1,2951  0,7554  1,1660
0,4316  1,6263  -0,0174  0,7802

K:
-0,3134  -1,2951  0,7554  -0,5830
0,4316  -1,6263  -0,0174  -0,3901

V:
0,6268  1,2951  -0,7554  -1,1660
-0,8631  1,6263  0,0174  -0,7802

q1 k2^T = (-0,3134)(0,4316) + (1,2951)(-1,6263) + (0,7554)(-0,0174) + (1,166)(-0,3901) = -2,7095
s12 = -2,7095 / 2 = -1,3548
a11 = 0,4299 / (0,4299 + 0,258) = 0,6249

S:
-0,8441  -1,3548
-1,3548  -1,3813

A:
0,6249  0,3751
0,5066  0,4934

MHA:
0,0679  1,4193  -0,4655  -1,0213
-0,1083  1,4585  -0,3741  -0,9756

x1:
0,3589  1,3971  0,4732  0,1098
1,0637  1,4015  0,3632  0,1555

LN_x1:
-0,4639  1,6680  -0,2292  -0,9754
0,6272  1,2942  -0,7558  -1,1658

MLP contoh: GELU(-0,4639) + 0,5 GELU(0,4639) = -0,1491 + 0,5(0,3148) = 0,0

In [11]:
print("5. BLOK DECODER UNTUK PENANDA AWAL")
for k in ["y", "LN_y", "v", "y1", "q", "s", "a", "ca", "y2", "mlp", "y3"]:
    tampil(k, dec[k])
fr = int(np.argmax(dec["a"][0])) + 1
print(f"Bobot {dec['a'][0].max()} berarti penanda awal paling banyak memperhatikan frame ke-{fr}.\n".replace(".", ",", 1))

print("6. KELUARAN")
L = dec["LN_y3"][0]
print(("z_main = " + " + ".join(f"({a})({b:g})" for a, b in zip(L, W_EMB[1])) + f" = {dec['z'][0, 1]}").replace(".", ","))
tampil("LN(y3)", dec["LN_y3"]); tampil("z", dec["z"]); tampil("p = softmax(z)", dec["p"])
print(pd.Series(dec["p"][0], index=TOKEN))

5. BLOK DECODER UNTUK PENANDA AWAL
y:
0,0000  0,5000  1,0000  0,5000

LN_y:
-1,4140  0,0000  1,4140  0,0000

v:
1,4140  0,0000  -1,4140  0,0000

y1:
1,4140  0,5000  -0,4140  0,5000

q:
0,7071  0,0000  -1,4142  0,0000

s:
0,2661  0,6892

a:
0,3958  0,6042

ca:
0,0890  -1,6126  0,7601  0,7636

y2:
1,5030  -1,1126  0,3461  1,2636

mlp:
0,7365  0,6487  -0,0241  0,4891

y3:
2,2395  -0,4639  0,3220  1,7527

Bobot 0,6042 berarti penanda awal paling banyak memperhatikan frame ke-2.

6. KELUARAN
z_main = (1,178)(1) + (-1,316)(0) + (-0,591)(-0,5) + (0,7289)(0) = 1,4735
LN(y3):
1,1780  -1,3160  -0,5910  0,7289

z:
0,3644  1,4735  0,6560  -1,7690

p = softmax(z):
0,1822  0,5523  0,2439  0,0216

SOT       0.1822
main      0.5523
memory    0.2439
EOT       0.0216
dtype: float64


In [12]:
print("7. BEAM SEARCH (lebar 2, dua langkah)")
p1 = dec["p"][0]
print("Langkah 1 menyimpan:", ", ".join(f"{t} (ln {p1[TOKEN.index(t)]} = {bulat(math.log(p1[TOKEN.index(t)]))})" for t in simpan).replace(".", ","))
display(tabel_beam)
urut = tabel_beam.sort_values("skor", ascending=False)
print("Isi beam:", ", ".join(f"{d} ({s})" for d, s in zip(urut.deret[:2], urut.skor[:2])).replace(".", ","))
keluaran = urut.deret.iloc[0]
assert keluaran == "main memory", f"keluaran model contoh menjadi '{keluaran}'"
print("Keluaran:", keluaran)

simpan_json = lambda d: {k: np.asarray(v).tolist() for k, v in d.items()}
(MAN / "langkah_1_7.json").write_text(json.dumps({
    "X": X.tolist(), "W_emb": W_EMB.tolist(), "W_emb_lama": W_EMB_LAMA.tolist(),
    "encoder": simpan_json(enc), "decoder": simpan_json(dec), "beam_disimpan": simpan, "keluaran": keluaran,
}, indent=1), encoding="utf-8")
tabel_beam.to_csv(MAN / "beam_search.csv", index=False, encoding="utf-8-sig")

7. BEAM SEARCH (lebar 2, dua langkah)
Langkah 1 menyimpan: main (ln 0,5523 = -0,5937), memory (ln 0,2439 = -1,411)


,deret,log_p_1,p_2,log_p_2,skor
0,main SOT,-0.5937,0.1810,-1.7093,-2.3030
1,main main,-0.5937,0.1098,-2.2091,-2.8028
2,main memory,-0.5937,0.5433,-0.6101,-1.2038
3,main EOT,-0.5937,0.1659,-1.7964,-2.3901
4,memory SOT,-1.4110,0.1890,-1.6660,-3.0770
5,memory main,-1.4110,0.3504,-1.0487,-2.4597
6,memory memory,-1.4110,0.4150,-0.8795,-2.2905
7,memory EOT,-1.4110,0.0456,-3.0878,-4.4988


Isi beam: main memory (-1,2038), memory memory (-2,2905)
Keluaran: main memory


## 5. Langkah 8–11: normalisasi, penyelarasan, WER, CER, TTER
Transkrip acuan diambil dari master AOK-04 pada segmen yang sama dengan spektrogram X. Transkrip hipotesis adalah ilustrasi yang memuat keluaran *main memory*, sebab keluaran varian sebenarnya pada segmen ini tidak memuat kesalahan istilah. Normalisasi, penandaan glosarium, dan penyelarasan memakai modul yang sama dengan Tahap 1 sampai Tahap 3.

In [13]:
master = pd.read_csv(MASTER / f"{FID}.tsv", sep="\t", dtype=str, keep_default_na=False, encoding="utf-8-sig")
baris = master[(master.start.astype(float).round(2) == MULAI) & (master.end.astype(float).round(2) == SELESAI)]
assert len(baris) == 1, f"segmen {MULAI}–{SELESAI} tidak ditemukan tepat satu kali"
ACUAN = baris.teks_acuan.iloc[0]
HIPOTESIS = "Yaitu mekanisme untuk mempercepat akses pembacaan data dari main memory ke dalam sepu."   # ilustrasi

kamus = pt.muat_kamus(KAMUS_PATH)
glosarium = pt.muat_glosarium(GLOSARIUM)
norm = pt.Normalisasi(kamus, glosarium[MATA_KULIAH])
ref, hyp = norm(ACUAN), norm(HIPOTESIS)
G = pt.himpunan_g(pt.tandai_glosarium(ref, glosarium[MATA_KULIAH]))

print("8. NORMALISASI")
display(pd.DataFrame({"sebelum": [ACUAN, HIPOTESIS], "sesudah": [" ".join(ref), " ".join(hyp)]},
                     index=["acuan", "hipotesis"]))
print(f"N = {len(ref)}, G = {{{', '.join(str(i + 1) for i in G)}}}  (posisi mulai dari 1)")

8. NORMALISASI


,sebelum,sesudah
acuan,Yaitu suatu mekanisme untuk mempercepat akses ...,yaitu suatu mekanisme untuk mempercepat akses ...
hipotesis,Yaitu mekanisme untuk mempercepat akses pembac...,yaitu mekanisme untuk mempercepat akses pembac...


N = 13, G = {9, 10, 13}  (posisi mulai dari 1)


In [14]:
import jiwer

print("9. PENYELARASAN DAN WER")
ops = mt.selaraskan(ref, hyp)
tabel_ops = pd.DataFrame([{
    "posisi_acuan": o.i_src + 1 if o.jenis != "I" else "--",
    "acuan": ref[o.i_src] if o.jenis != "I" else "--",
    "hipotesis": hyp[o.i_res] if o.i_res is not None else "--",
    "operasi": "" if o.jenis == "C" else (f"I (i_src = {o.i_src + 1})" if o.jenis == "I" else o.jenis),
    "istilah_G": "✓" if o.jenis != "I" and o.i_src in G else "",
} for o in ops])
display(tabel_ops)
S, D, I = (sum(o.jenis == j for o in ops) for j in "SDI")
N = len(ref)
wer = bulat((S + D + I) / N)
cek = mt.periksa_jiwer(ref, hyp, ops)
assert cek["sama"], cek
print(f"WER = (S + D + I) / N = ({S} + {D} + {I}) / {N} = {wer}".replace(".", ","))

print("\n10. CER")
c = jiwer.process_characters(" ".join(ref), " ".join(hyp))
NC = len(" ".join(ref))
cer = bulat((c.substitutions + c.deletions + c.insertions) / NC)
print(f"N_C = {NC} karakter, termasuk {' '.join(ref).count(' ')} spasi")
print(f"CER = ({c.substitutions} + {c.deletions} + {c.insertions}) / {NC} = {cer}".replace(".", ","))

print("\n11. TTER")
EG = [o for o in ops if o.jenis != "C" and (o.i_src in G or (o.i_src >= N and (N - 1) in G))]
tter = bulat(len(EG) / len(G))
print("Operasi yang jatuh di G:", [(o.jenis, o.i_src + 1) for o in EG])
print(f"TTER = |E_G| / |G| = {len(EG)} / {len(G)} = {tter}".replace(".", ","))

tabel_ops.to_csv(MAN / "penyelarasan.csv", index=False, encoding="utf-8-sig")
metrik = {"acuan": ACUAN, "hipotesis": HIPOTESIS, "token_acuan": ref, "token_hipotesis": hyp,
          "G": [i + 1 for i in G], "S": S, "D": D, "I": I, "N": N, "WER": wer,
          "S_C": c.substitutions, "D_C": c.deletions, "I_C": c.insertions, "N_C": NC, "CER": cer,
          "E_G": len(EG), "TTER": tter}
_ = (MAN / "metrik.json").write_text(json.dumps(metrik, indent=2, ensure_ascii=False), encoding="utf-8")

9. PENYELARASAN DAN WER


,posisi_acuan,acuan,hipotesis,operasi,istilah_G
0,1,yaitu,yaitu,,
1,2,suatu,--,D,
2,3,mekanisme,mekanisme,,
3,4,untuk,untuk,,
4,5,mempercepat,mempercepat,,
5,6,akses,akses,,
6,7,pembacaan,pembacaan,,
7,--,--,data,I (i_src = 8),
8,8,dari,dari,,
9,9,main,main,,✓


WER = (S + D + I) / N = (1 + 1 + 1) / 13 = 0,2308

10. CER
N_C = 85 karakter, termasuk 12 spasi
CER = (1 + 6 + 6) / 85 = 0,1529

11. TTER
Operasi yang jatuh di G: [('S', 13)]
TTER = |E_G| / |G| = 1 / 3 = 0,3333


## 6. Langkah 12: RTF
Waktu proses diambil dari log Tahap 1 untuk rekaman AOK-04 pada varian terpilih, sehingga contoh RTF memakai angka pengukuran sebenarnya. Pada pengukuran resmi, waktu proses dan durasi dijumlahkan atas seluruh rekaman uji lebih dahulu (subbab 4.9).

In [15]:
waktu = pd.DataFrame([json.loads(b) for b in LOG_WAKTU.read_text(encoding="utf-8").splitlines() if b.strip()])
w = waktu[(waktu.konfigurasi == VARIAN_RTF) & (waktu.file_id == FID)].sort_values("ulangan")
assert len(w) == 3, f"ulangan tercatat: {len(w)}"
durasi = float(w.durasi_s.iloc[0])
rtf = pd.DataFrame({"ulangan": w.ulangan.values, "waktu_proses_s": w.proses_s.values,
                    "durasi_s": durasi, "RTF": [bulat(t / durasi) for t in w.proses_s]})
rerata = bulat(rtf.RTF.mean())
display(rtf)
for _, b in rtf.iterrows():
    print(f"RTF_{int(b.ulangan)} = {b.waktu_proses_s} / {durasi} = {b.RTF}".replace(".", ","))
print(f"rerata RTF = {rerata} → sekitar {round(rerata * 60, 1)} detik proses untuk setiap menit audio".replace(".", ","))
rtf.to_csv(MAN / "rtf.csv", index=False, encoding="utf-8-sig")

,ulangan,waktu_proses_s,durasi_s,RTF
0,1,181.529,1171.989,0.1549
1,2,164.919,1171.989,0.1407
2,3,159.957,1171.989,0.1365


RTF_1 = 181,529 / 1171,989 = 0,1549
RTF_2 = 164,919 / 1171,989 = 0,1407
RTF_3 = 159,957 / 1171,989 = 0,1365
rerata RTF = 0,144 → sekitar 8,6 detik proses untuk setiap menit audio


## 7. Metadata run

In [16]:
import importlib.metadata as im
def versi(p):
    try:
        return im.version(p)
    except im.PackageNotFoundError:
        return None

meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "Manualisasi (subbab 4.11)",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "spektrogram": {k: info_x[k] for k in ["rekaman", "segmen_detik", "ukuran_mel", "pita", "frame_mulai", "detik_mulai"]},
    "X": X.tolist(),
    "W_emb": {"lama": W_EMB_LAMA.tolist(), "baru": W_EMB.tolist()},
    "keluaran_langkah_7": keluaran,
    "metrik": {"WER": wer, "CER": cer, "TTER": tter, "RTF_rerata": rerata, "varian_RTF": VARIAN_RTF},
    "kamus": {"status": kamus.status, "tanggal_kunci": kamus.tanggal_kunci},
    "versi_pustaka": {p: versi(p) for p in ["faster-whisper", "jiwer", "numpy", "pandas", "soundfile"]},
}
(MAN / "run_meta_manualisasi.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(meta["metrik"], indent=2))

{
  "WER": 0.2308,
  "CER": 0.1529,
  "TTER": 0.3333,
  "RTF_rerata": 0.144,
  "varian_RTF": "large-v3"
}


## Output

| File di `data/output/manualisasi/` | Isi | Dipakai di |
|---|---|---|
| `X_spektrogram.json` | Asal spektrogram X: rekaman, segmen, pita, frame, nilai 4 desimal dan nilai yang dibulatkan | Subbab 4.11 langkah 1, lampiran |
| `langkah_1_7.json` | Seluruh matriks antara langkah 1–7 beserta bobot *embedding* lama dan baru | Subbab 4.11 langkah 1–7 |
| `beam_search.csv` | Skor *beam search* langkah kedua | Tabel 4.4 |
| `penyelarasan.csv` | Penyelarasan transkrip acuan dan hipotesis | Tabel 4.6 |
| `metrik.json` | Teks sebelum dan sesudah normalisasi, G, WER, CER, TTER | Tabel 4.5, langkah 8–11 |
| `rtf.csv` | Waktu proses dan RTF tiap ulangan AOK-04 | Langkah 12 |
| `run_meta_manualisasi.json` | Parameter dan versi pustaka | Lampiran |